# 03 · Split 70 / 20 / 10 and check the data (MedSafe-AI · Component 3)

We split the drug pairs into **train 70 %** (the model learns), **validation 20 %** (choose settings and the score cut-off) and **test 10 %** (final exam, used once).

Two splits are made:
* **`pair`** – random split of drug pairs. A test pair is new, but its drugs usually appeared in other training pairs.
* **`unseen_drug`** – harder and more honest for a new medicine: some drugs are kept **out of training completely**. Every validation/test pair contains at least one drug the model has never seen.

**Mistakes from the earlier (friend's) notebooks that this notebook avoids**
| Earlier problem | What we do instead |
|---|---|
| Data truncated (Excel) or synthetic | Full official TWOSIDES/OFFSIDES, cleaned in pieces |
| Every test drug was forced into training (no unseen drugs) | We report how many test drugs are unseen, and we also have the `unseen_drug` split |
| Only drug IDs as features | Chemical fingerprints + drug class |
| Looked only at P@k and "accuracy" | AUPRC, P@k, F1, High-severity recall, always next to a baseline |
| Possible leakage from labels into features | Drug-history features use **training pairs only**, with leave-one-out; plus a label-shuffle test in notebook 04 |

In [ ]:
import sys, importlib.util, subprocess
from pathlib import Path

HERE = Path.cwd()                                   # the folder of this notebook (component3)
if not (HERE / "c3_utils.py").exists():
    if Path("/content").exists():                   # Google Colab: get the code from GitHub
        REPO_DIR = Path("/content/repo")
        if not (REPO_DIR / "component3" / "c3_utils.py").exists():
            subprocess.run(["rm", "-rf", str(REPO_DIR)])
            subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/sksivakajan/mac-doc.git", str(REPO_DIR)])
        HERE = REPO_DIR / "component3"
        import os; os.chdir(HERE)
        print("Colab: code taken from GitHub ->", HERE)
    else:                                           # Jupyter was started from another folder
        HERE = Path(r"D:\med\data set\component3")
sys.path.insert(0, str(HERE))

def need(*pairs):
    """Install the packages that are missing. pairs = (import name, pip name)."""
    missing = [pip for mod, pip in pairs if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", missing)
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        except Exception as e:
            print("!! Could not install", missing, "->", e)

need(("pandas", "pandas"), ("numpy", "numpy"), ("matplotlib", "matplotlib"))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import c3_utils as c3

WORK = HERE / "work"
pairs = pd.read_csv(WORK / "modeling_pairs.csv")
Y = np.load(WORK / "modeling_Y.npy")
effects = pd.read_csv(WORK / "modeling_effects.csv")
ia, ib = pairs["idx_a"].to_numpy(), pairs["idx_b"].to_numpy()
print(f"{len(pairs):,} drug pairs, {Y.shape[1]} side effects, {len(set(ia) | set(ib)):,} drugs")

## Step 1 · Make the splits

In [ ]:
FRACS = (0.70, 0.20, 0.10)
SEED = c3.SEED

splits = {
    "pair": c3.split_random(len(pairs), FRACS, SEED),
    "unseen_drug": c3.split_unseen_drug(ia, ib, FRACS, SEED),
}
for mode, (tr, va, te) in splits.items():
    n = len(pairs)
    print(f"{mode:12s} train {len(tr):>7,} ({100*len(tr)/n:4.1f}%)   val {len(va):>7,} ({100*len(va)/n:4.1f}%)   test {len(te):>7,} ({100*len(te)/n:4.1f}%)")
    np.savez(WORK / f"split_{mode}.npz", train=tr, val=va, test=te)

## Step 2 · Check the data after the split
All of these must look right **before** you train anything.

In [ ]:
reports = {}
for mode, parts in splits.items():
    rep = c3.check_split(ia, ib, Y, parts, effects)
    reports[mode] = rep
    print("=" * 70)
    print("SPLIT:", mode)
    print("-- 1. Same pair in two parts? (must all be 0)")
    print("  ", rep["pair_overlap"])
    print("-- 2. Duplicate pairs in the whole table (must be 0):", rep["duplicate_pairs_total"])
    print("-- 3. Drugs that were never in TRAINING")
    display(pd.DataFrame(rep["drug_overlap"]).set_index("part"))
    print("-- 4. Labels in each part")
    display(pd.DataFrame(rep["labels"]).set_index("part"))
    for part, names in rep["zero_pos_effects"].items():
        if names:
            print(f"  !! side effects with NO positive pair in {part}: {names[:10]}")
    assert all(v == 0 for v in rep["pair_overlap"].values()), "A pair is in two parts!"
    assert rep["duplicate_pairs_total"] == 0, "Duplicate pairs found!"
print("\nAll hard checks passed.")

### How to read the checks
* **Check 1 and 2** must be zero. If not, the test score would be too good (the model saw the answer).
* **Check 3**: in the `pair` split a few validation/test pairs may contain a drug that is not in training. In the `unseen_drug` split this is 100 % by design. Report **both** results in the thesis.
* **Check 4**: the share of positives should be similar in all parts. In the `unseen_drug` split it can differ a little because different drugs have different side-effect profiles.

In [ ]:
# Easy baseline: "always predict the most common side effects". Any model must beat this.
for mode, (tr, va, te) in splits.items():
    prev = Y[tr].mean(0)
    S = np.tile(prev, (len(te), 1))
    r = c3.evaluate(Y[te], S, c3.best_threshold(Y[va], np.tile(prev, (len(va), 1))))
    print(f"{mode:12s} baseline on TEST:  micro-AUPRC {r['micro_AUPRC']:.3f}   P@3 {r['P@3']:.3f}   P@10 {r['P@10']:.3f}")

# plot: how common is each side effect in train / val / test (pair split)
tr, va, te = splits["pair"]
fig, ax = plt.subplots(figsize=(11, 3.2))
x = np.arange(Y.shape[1])
for ix, nm, col in [(tr, "train", "#3b6ea5"), (va, "val", "#e0a030"), (te, "test", "#c0504d")]:
    ax.plot(x, Y[ix].mean(0), label=nm, color=col, lw=1.4)
ax.set_xlabel("side effect (most common first)"); ax.set_ylabel("share of pairs"); ax.legend(); ax.set_title("Side-effect frequency per part (pair split)")
plt.tight_layout(); plt.show()

## Done ✔
Saved `split_pair.npz` and `split_unseen_drug.npz` in the `work` folder.

**Next:** open `04_train_xgboost_rf_svm.ipynb`.